STEP 1: Import Required Libraries

In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
from sklearn.cluster import KMeans, DBSCAN, MeanShift, estimate_bandwidth
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples
from skimage import segmentation, color
from skimage.util import img_as_float
from skimage.segmentation import mark_boundaries
import matplotlib.cm as cm

STEP 2: Load Image

In [3]:
def gdrive_to_direct_url(share_url):
    """Convert Google Drive shareable URL to direct download URL"""
    file_id = share_url.split("/d/")[1].split("/")[0]
    return f"https://drive.google.com/uc?export=download&id={file_id}"

image_url = "https://drive.google.com/file/d/1mlHMl6_7fnNh3Z2PDyWwL9z3FXLr2hXD/view?usp=sharing"
direct_url = gdrive_to_direct_url(image_url)

# Load image using OpenCV
resp = urllib.request.urlopen(direct_url)
image_np = np.asarray(bytearray(resp.read()), dtype="uint8")
image = cv2.imdecode(image_np, cv2.IMREAD_COLOR)

if image is None:
    raise ValueError("Image could not be loaded. Check the URL!")

# Resize for computational efficiency
image = cv2.resize(image, (256, 256))

STEP 3: Preprocessing (Sharpening + Conversion)

In [4]:
# Sharpen the image to enhance edges and improve clustering
kernel_sharp = np.array([[0, -1, 0],
                         [-1, 5, -1],
                         [0, -1, 0]])
image = cv2.filter2D(image, -1, kernel_sharp)

# Convert image to RGB and normalize to float [0,1]
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image_float = img_as_float(image_rgb)

# Flatten image to (num_pixels x 3) array
pixels = image_rgb.reshape(-1, 3)

STEP 4: Utility Functions for Visualization

In [5]:
def visualize_clusters(labels, title, img_shape):
    """Display original + clustered boundary image"""
    segmented = labels.reshape(img_shape[:2])
    boundaries = mark_boundaries(image_float, segmented)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1); plt.imshow(image_rgb); plt.title("Original Image")
    plt.subplot(1, 2, 2); plt.imshow(boundaries); plt.title(title + " + Boundaries")
    plt.tight_layout(); plt.show()

    return segmented

def print_metrics(labels, title):
    """Print silhouette score (if meaningful)"""
    if len(np.unique(labels)) > 1:
        score = silhouette_score(pixels, labels)
        print(f"✅ {title} Silhouette Score: {score:.4f}")
    else:
        print(f"⚠️ {title} did not form enough clusters.")

def plot_silhouette(X, labels, title):
    """Silhouette plot per cluster"""
    if len(np.unique(labels)) <= 1:
        print(f"⚠️ Skipping silhouette plot for {title} (not enough clusters).")
        return

    silhouette_vals = silhouette_samples(X, labels)
    silhouette_avg = np.mean(silhouette_vals)
    n_clusters = len(np.unique(labels))

    y_lower = 10
    plt.figure(figsize=(8, 4))
    for i in range(n_clusters):
        ith_vals = silhouette_vals[labels == i]
        ith_vals.sort()
        size_i = ith_vals.shape[0]
        y_upper = y_lower + size_i
        color = cm.nipy_spectral(float(i) / n_clusters)
        plt.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_vals, facecolor=color, edgecolor=color, alpha=0.7)
        plt.text(-0.05, y_lower + 0.5 * size_i, str(i))
        y_lower = y_upper + 10

    plt.axvline(x=silhouette_avg, color="red", linestyle="--", label="Avg Score")
    plt.xlabel("Silhouette Coefficient Values")
    plt.ylabel("Cluster Label")
    plt.title(f"Silhouette Plot for {title}")
    plt.legend()
    plt.show()

# ----------------------------------------
# 🔵 K-MEANS CLUSTERING
# ----------------------------------------
# 📌 Definition: Partitions data into k groups by minimizing variance
# 📘 Formula: min ∑ ||xᵢ - μ_j||²  (μ_j = cluster center)

kmeans = KMeans(n_clusters=4, random_state=0)
k_labels = kmeans.fit_predict(pixels)
visualize_clusters(k_labels, "KMeans Clustering", image.shape)
print_metrics(k_labels, "KMeans")
plot_silhouette(pixels, k_labels, "KMeans")

# ----------------------------------------
# 🔵 GAUSSIAN MIXTURE MODEL (GMM)
# ----------------------------------------
# 📌 Definition: Models data as a mixture of Gaussians
# 📘 Formula: log-likelihood ∑ log ∑ π_k N(xᵢ | μ_k, Σ_k)

gmm = GaussianMixture(n_components=4, covariance_type='tied', random_state=0)
g_labels = gmm.fit_predict(pixels)
visualize_clusters(g_labels, "GMM Clustering", image.shape)
print_metrics(g_labels, "GMM")
plot_silhouette(pixels, g_labels, "GMM")

# ----------------------------------------
# 🔵 MEANSHIFT CLUSTERING
# ----------------------------------------
# 📌 Definition: Shifts each point toward the mean of its neighborhood
# 📘 Formula: m(x) = ∑ xᵢ K(x - xᵢ) / ∑ K(x - xᵢ)

bandwidth = estimate_bandwidth(pixels, quantile=0.1, n_samples=1000)
ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
ms_labels = ms.fit_predict(pixels)
visualize_clusters(ms_labels, "MeanShift Clustering", image.shape)
print_metrics(ms_labels, "MeanShift")
plot_silhouette(pixels, ms_labels, "MeanShift")

# ----------------------------------------
# 🔵 DBSCAN CLUSTERING
# ----------------------------------------
# 📌 Definition: Finds dense regions separated by low-density areas
# 📘 Formula: A point is core if ≥ min_samples in ε-neighborhood

db = DBSCAN(eps=6, min_samples=50)
db_labels = db.fit_predict(pixels)
visualize_clusters(db_labels, "DBSCAN Clustering", image.shape)
print_metrics(db_labels, "DBSCAN")
plot_silhouette(pixels, db_labels, "DBSCAN")

Output hidden; open in https://colab.research.google.com to view.